# 演算子を挿入して99を構成する ― 括弧の付け方を Z3 に探索させます

## 本章で学ぶこと

数値の並びは固定されており、その間に `+`, `-`, `*` を挿入し、括弧によって計算順序を変更する問題です。

- 「隣接する2要素を演算して1要素に置き換える」操作を繰り返すことで、任意の括弧の付け方を表現できます
- 各段階の配列を Z3 の変数とし、`If` で演算を選択します
- 最後に、得られた手順を数式の文字列に復元します

数の順序が任意である「24を作る」問題との差異も明確になります。

> 次の式において、`?` に `+`, `-`, `*` のいずれかを挿入し、括弧を適切に配置して式の値が 99 となるようにしてください。
>
> `1 ? 2 ? 3 ? 4 ? 5 ? 6 ? 7 ? 8 ? 9 = 99`

## 方針

数を逐次計算するのではなく、**隣接する2要素を演算して1要素に統合する**操作を繰り返します。

例えば `1 2 3 4 5` から開始し、

1. `3 + 4` を演算 → `1 2 7 5`
2. `1 * 2` を演算 → `2 7 5`
3. `7 + 5` を演算 → `2 12`
4. `2 - 12` を演算 → `-10`

このように、各段階で要素数が1つずつ減少します。最終的に残った1つの要素が式の値となります。

```{mermaid}
graph
a1[1]
a2[2]
a3[3+4]
a4[5]
b1[1*2]
b2[3+4]
b3[5]
c1[1*2]
c2[3+4-5]
d1[1*2+3+4-5]
1 --> a1
2 --> a2
3 --> a3
4 --> a3
5 --> a4
a1 --> b1
a2 --> b1
a3 --> b2
a4 --> b3
b1 --> c1
b2 --> c2
b3 --> c2
c1 --> d1
c2 --> d1
```

この「どの位置をどの演算子で統合するか」を Z3 に決定させれば、括弧の付け方を全探索した場合と等価になります。段階数は $n-1$ 回であり、各段階で選択する位置は「その時点の要素数 − 1」通りです。

## 段階ごとの計算

段階 `l` の配列を `f[l]` とします。初期段階 `f[0]` は `[1, 2, …, 9]` であり、段階が進むごとに要素数が1つ減少します。

`calc_tree(f, locs, ops)` は、各段階の計算を制約として列挙します。段階 `l` で位置 `p = locs[l-1]` を選択すると、

- `i < p` … `f[l][i] == f[l-1][i]`（そのまま保持されます）
- `i == p` … `f[l][p] == f[l-1][p] op f[l-1][p+1]`（ここで演算します）
- `i > p` … `f[l][i] == f[l-1][i+1]`（左側から1つずれて保持されます）

In [1]:
from z3 import *
from helper.z3 import all_solutions

OP_SYMBOLS = '+-*'   # 0:+, 1:-, 2:*


def calc_op(x1, x2, op):
    """op の番号に応じて、足す・引く・掛ける を選ぶ"""
    return If(op == 0, x1 + x2, If(op == 1, x1 - x2, x1 * x2))


def calc_tree(f, locs, ops):
    """各段階の計算を制約のリストにして返す"""
    exprs = []
    for l in range(1, len(f)):
        prev = f[l - 1]
        for i in range(len(f[l])):
            exprs.extend([
                Implies(i < locs[l - 1], f[l][i] == prev[i]),
                Implies(i == locs[l - 1], f[l][i] == calc_op(prev[i], prev[i + 1], ops[l - 1])),
                Implies(i > locs[l - 1], f[l][i] == prev[i + 1]),
            ])
    return exprs

### If による演算子の選択

```python
If(op == 0, x1 + x2, If(op == 1, x1 - x2, x1 * x2))
```

`If(条件, 真のときの値, 偽のときの値)` を入れ子にして、3種類の演算を切り替えています。

| `op` | 計算 |
|---|---|
| 0 | `x1 + x2` |
| 1 | `x1 - x2` |
| 2 | `x1 * x2` |

条件が増えると入れ子が深くなるため、選択肢が多い場合は `Function` を用いる方が可読性が高いです（本章では選択肢が3つであるため `If` で十分です）。

## 数式としての表示

Z3 が出力するのは数値のみであるため、元の数式に復元する仕組みを用意します。`Expr` は「左の式、演算子、右の式」を保持する小さなクラスであり、`output()` が段階的な手順をたどって文字列を組み立てます。

`*` や `-` の右側に `+` や `-` が現れる場合は括弧を付与します。`1 + 2 * 3` と `(1 + 2) * 3` を混同しないようにするためです。

In [2]:
class Expr:
    """計算の木を、括弧付きの文字列にする"""
    def __init__(self, left, op, right):
        self.left = left
        self.op = op
        self.right = right

    def __str__(self):
        # 左側：* の左に + - が来るなら括弧が必要
        if isinstance(self.left, Expr) and self.op == '*' and self.left.op in '+-':
            left = f'({self.left})'
        else:
            left = f'{self.left}'

        # 右側：* や - の右に + - が来るなら括弧が必要
        if isinstance(self.right, Expr) and self.op in '*-' and self.right.op in '+-':
            right = f'({self.right})'
        else:
            right = f'{self.right}'

        return f'{left} {self.op} {right}'


def output(numbers, locs, ops):
    """数の並びと手順から、数式の文字列を作る"""
    exprs = [str(x) for x in numbers]
    for loc, op in zip(locs, ops):
        exprs[loc] = Expr(exprs[loc], OP_SYMBOLS[op], exprs[loc + 1])
        del exprs[loc + 1]
    return str(exprs[0])

## 小規模例による動作確認

`1 2 3 4 5` に対して、固定した手順を適用します。

- 第1段階：位置2 に `+` を適用します（`3 + 4`）
- 第2段階：位置0 に `*` を適用します（`1 * 2`）
- 第3段階：位置1 に `-` を適用します（`7 - 5`）
- 第4段階：位置0 に `+` を適用します（`2 + 2`）

結果は `1 * 2 + ((3 + 4) - 5) = 4` となります。

In [3]:
n = 5
f = [[Int(f'f{l}_{i}') for i in range(n - l)] for l in range(n)]

solver = Solver()
for i in range(n):
    solver.add(f[0][i] == i + 1)      # 最初は 1, 2, 3, 4, 5

locs = [2, 0, 1, 0]                   # どの位置をまとめるか
ops = [0, 2, 1, 0]                    # +, *, -, +
solver.add(calc_tree(f, locs, ops))

assert solver.check() == sat
m = solver.model()

expr = output([i + 1 for i in range(n)], locs, ops)
print(expr, "=", eval(expr))

print("各段階の値")
for level in range(n):
    print([m[f[level][i]] for i in range(n - level)])

1 * 2 + 3 + 4 - 5 = 4
各段階の値
[1, 2, 3, 4, 5]
[1, 2, 7, 5]
[2, 7, 5]
[2, 2]
[4]


手計算のとおり、`1 * 2 + 3 + 4 - 5 = 4` となります（計算木としては `1 * 2 + ((3 + 4) - 5)` です）。各段階の値からも、要素数が1つずつ減少する様子が確認できます。

## 99となる組み合わせの探索

次に、位置と演算子を Z3 に決定させます。9個の数であるため、制約は「位置 `locs[i]` は 0 以上 `9-i-1` 未満」「演算子は 0〜2」とします。

In [4]:
n = 9
f = [[Int(f'f{l}_{i}') for i in range(n - l)] for l in range(n)]

solver = Solver()
for i in range(n):
    solver.add(f[0][i] == i + 1)          # 1 から 9

locs = IntVector('loc', n - 1)            # 各段階でまとめる位置
ops = IntVector('op', n - 1)              # 各段階の演算子

for i, loc in enumerate(locs):
    solver.add(0 <= loc, loc < n - i - 1)  # その段階の要素数 - 1 まで

for op in ops:
    solver.add(0 <= op, op <= 2)           # 0:+, 1:-, 2:*

solver.add(calc_tree(f, locs, ops))
solver.add(f[-1][0] == 99)                 # 最後に残った数が 99

print(solver.check())

m = solver.model()
print("各段階の値")
for level in range(n):
    print([m[f[level][i]] for i in range(n - level)])

locs_used = [m[x].as_long() for x in locs]
ops_used = [m[x].as_long() for x in ops]
print("位置:", locs_used)
print("演算子:", [OP_SYMBOLS[x] for x in ops_used])

sat
各段階の値
[1, 2, 3, 4, 5, 6, 7, 8, 9]
[1, 2, -1, 5, 6, 7, 8, 9]
[1, 2, -1, 5, 6, 7, 17]
[1, 2, -1, 5, 6, 119]
[1, 2, -1, 5, -113]
[1, 3, 5, -113]
[1, 15, -113]
[1, -98]
[99]
位置: [2, 6, 5, 4, 1, 1, 1, 0]
演算子: ['-', '+', '*', '-', '-', '*', '+', '-']


最後に、得られた手順を数式に復元します。

In [5]:
expr = output(list(range(1, 10)), locs_used, ops_used)
print(expr, "=", eval(expr))

1 - ((2 - (3 - 4)) * 5 + 6 - 7 * (8 + 9)) = 99


例えば、次の式が得られます（実行ごとに異なる式が得られる場合があります）。

```
1 - ((2 - (3 - 4)) * 5 + 6 - 7 * (8 + 9)) = 99
```

Z3 が探索するのは**条件を満たす最初の解**であるため、式の形は毎回同一とは限りません。いずれも 99 となります。

## まとめ

- 固定された数の並びに対する括弧の付け方は、「隣接する2要素を統合する」操作列で表現できます
- 段階ごとに配列を用意し、`calc_tree()` で「保持・演算・ずれ」の3通りを `Implies` で記述します
- 演算子は整数の番号とし、`If` で演算を切り替えます
- 得られた手順は、計算木（`Expr`）に復元すれば数式として読めます
- 数の順序も任意である「24を作る」問題では、順列の制約がさらに1つ増えます（前章）